In [7]:
import kagglehub

path = kagglehub.dataset_download("salviohexia/isic-2019-skin-lesion-images-for-classification")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'isic-2019-skin-lesion-images-for-classification' dataset.
Path to dataset files: /kaggle/input/isic-2019-skin-lesion-images-for-classification


In [8]:
!pip install torchmetrics

In [9]:
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b0
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from tqdm import tqdm
import torchmetrics
from torch.utils.data import Subset
import numpy as np

In [10]:
train_transform = transforms.Compose([
    transforms.Resize((260, 260)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(180),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [11]:
test_transform = transforms.Compose([
    transforms.Resize((260, 260)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [12]:
dataset = datasets.ImageFolder(root="/kaggle/input/isic-2019-skin-lesion-images-for-classification")

In [13]:
dataset.class_to_idx

{'AK': 0, 'BCC': 1, 'BKL': 2, 'DF': 3, 'MEL': 4, 'NV': 5, 'SCC': 6, 'VASC': 7}

In [14]:
dataset_size = len(dataset)
dataset_size

25331

In [15]:
indices = list(range(len(dataset)))
np.random.shuffle(indices)
split = int(0.8 * len(indices))
split

20264

In [16]:
# train_dataset_size = int(0.8 * dataset_size)
# test_dataset_size = dataset_size - train_dataset_size

# print(f"Train data size : {train_dataset_size} and Test data size : {test_dataset_size}")

In [17]:
# from torch.utils.data import random_split
# train_dataset, test_dataset = random_split(dataset=dataset,lengths=[train_dataset_size,test_dataset_size])

In [18]:
# train_dataset.dataset.transform = train_transform
# test_dataset.dataset.transform = test_transform

In [19]:
data_path = "/kaggle/input/isic-2019-skin-lesion-images-for-classification"
train_dataset = Subset(datasets.ImageFolder(root=data_path, transform=train_transform), indices[:split])
test_dataset  = Subset(datasets.ImageFolder(root=data_path, transform=test_transform), indices[split:])

In [20]:
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=32)

In [21]:
device = ("cuda" if torch.cuda.is_available() else "cpu")
device

'cuda'

In [22]:
model = efficientnet_b0(weights="IMAGENET1K_V1")

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 55.9MB/s]


In [23]:
model.classifier

Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=1000, bias=True)
)

In [24]:
model.classifier[1]

Linear(in_features=1280, out_features=1000, bias=True)

In [25]:
in_features = model.classifier[1].in_features
in_features

1280

In [26]:
model.classifier[1] = nn.Linear(in_features=in_features,out_features=8)

In [27]:
model.classifier

Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=8, bias=True)
)

In [28]:
for name,param in model.features.named_children():
  print(f"Name : {name} and Param : {param}")

Name : 0 and Param : Conv2dNormActivation(
  (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): SiLU(inplace=True)
)
Name : 1 and Param : Sequential(
  (0): MBConv(
    (block): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): SiLU(inplace=True)
      )
      (1): SqueezeExcitation(
        (avgpool): AdaptiveAvgPool2d(output_size=1)
        (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
        (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
        (activation): SiLU(inplace=True)
        (scale_activation): Sigmoid()
      )
      (2): Conv2dNormActivation(
        (0): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (1):

In [29]:
for param in model.features.parameters():
  param.requires_grad = False

In [30]:
for name, param in model.features.named_children():
    if int(name) >= 6:  # Unfreeze blocks 6, 7, 8
        for p in param.parameters():
            p.requires_grad = True


In [31]:
for name,param in model.named_parameters():
  if param.requires_grad:
    print(name)

features.6.0.block.0.0.weight
features.6.0.block.0.1.weight
features.6.0.block.0.1.bias
features.6.0.block.1.0.weight
features.6.0.block.1.1.weight
features.6.0.block.1.1.bias
features.6.0.block.2.fc1.weight
features.6.0.block.2.fc1.bias
features.6.0.block.2.fc2.weight
features.6.0.block.2.fc2.bias
features.6.0.block.3.0.weight
features.6.0.block.3.1.weight
features.6.0.block.3.1.bias
features.6.1.block.0.0.weight
features.6.1.block.0.1.weight
features.6.1.block.0.1.bias
features.6.1.block.1.0.weight
features.6.1.block.1.1.weight
features.6.1.block.1.1.bias
features.6.1.block.2.fc1.weight
features.6.1.block.2.fc1.bias
features.6.1.block.2.fc2.weight
features.6.1.block.2.fc2.bias
features.6.1.block.3.0.weight
features.6.1.block.3.1.weight
features.6.1.block.3.1.bias
features.6.2.block.0.0.weight
features.6.2.block.0.1.weight
features.6.2.block.0.1.bias
features.6.2.block.1.0.weight
features.6.2.block.1.1.weight
features.6.2.block.1.1.bias
features.6.2.block.2.fc1.weight
features.6.2.blo

In [32]:
class_counts = torch.tensor([867, 3323, 2624, 239, 4522, 12875, 628, 253], dtype=torch.float)
class_weights = 1 / class_counts
class_weights = class_weights / class_weights.sum() * 8  #to normalize
class_weights

tensor([0.7778, 0.2029, 0.2570, 2.8215, 0.1491, 0.0524, 1.0738, 2.6654])

In [33]:
loss_fn = nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer = torch.optim.Adam([
    {"params":model.features.parameters(),"lr":1e-4},
    {"params":model.classifier.parameters(),"lr":1e-3}
    ]);
epochs = 30
num_classes = 8
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

In [34]:
# model = model.to(device)
# accuracy_metric = torchmetrics.Accuracy(task="multiclass",num_classes=num_classes).to(device)

# for epoch in range(epochs):
#   model.train()
#   total_loss = 0
#   accuracy_metric.reset()

#   for images,labels in tqdm(train_loader):
#     images = images.to(device)
#     labels = labels.to(device)

#     outputs = model(images)
#     loss = loss_fn(outputs,labels)

#     optimizer.zero_grad()
#     loss.backward()
#     optimizer.step()

#     total_loss += loss.item()
#     accuracy_metric.update(outputs,labels)

#   avg_epoch_loss = (total_loss / len(train_loader))
#   epoch_acc = accuracy_metric.compute()

#   print(f"Epoch {epoch} Loss : {avg_epoch_loss} Acc : {epoch_acc}")


In [35]:
# model.eval()
# acc_metric = torchmetrics.Accuracy("multiclass",num_classes=num_classes)
# test_loss = 0

# with torch.no_grad():
#   for images,labels in tqdm(test_loader):
#     images = images.to(device)
#     labels = labels.to(device)

#     outputs = model(images)
#     loss = loss_fn(outputs,labels)
#     test_loss += loss.item()

#     acc_metric.update(outputs,labels)

# avg_test_loss = (test_loss / len(test_loader))
# test_acc = acc_metric.compute()

# print(f"Test loss : {avg_test_loss} Test acc : {test_acc}")

In [ ]:
model = model.to(device)
train_accuracy_metric = torchmetrics.Accuracy(task="multiclass", num_classes=num_classes).to(device)
val_accuracy_metric = torchmetrics.Accuracy(task="multiclass", num_classes=num_classes).to(device)

best_val_acc = 0
patience = 5
patience_counter = 0

for epoch in range(epochs):
    model.train()
    total_loss = 0
    train_accuracy_metric.reset()

    for images, labels in tqdm(train_loader):
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = loss_fn(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        train_accuracy_metric.update(outputs, labels)

    avg_train_loss = total_loss / len(train_loader)
    train_acc = train_accuracy_metric.compute()

    model.eval()
    val_loss = 0
    val_accuracy_metric.reset()

    with torch.no_grad():
        for images, labels in tqdm(test_loader):
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = loss_fn(outputs, labels)
            val_loss += loss.item()
            val_accuracy_metric.update(outputs, labels)

    avg_val_loss = val_loss / len(test_loader)
    val_acc = val_accuracy_metric.compute()

    scheduler.step()

    print(f"Epoch {epoch} | Train Loss: {avg_train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {avg_val_loss:.4f} Val Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), 'best_model.pth')
        print(f"  ✓ Best model saved (val_acc: {val_acc:.4f})")
    else:
        patience_counter += 1
        print(f"  No improvement. Patience: {patience_counter}/{patience}")
        if patience_counter >= patience:
            print(f"Early stopping triggered at epoch {epoch}")
            break

model.load_state_dict(torch.load('best_model.pth'))
print(f"Best Val Acc: {best_val_acc:.4f}")

100%|██████████| 159/159 [01:52<00:00,  1.41it/s]


Epoch 0 | Train Loss: 1.5157 Acc: 0.5006 | Val Loss: 1.2064 Val Acc: 0.5974
  ✓ Best model saved (val_acc: 0.5974)


100%|██████████| 159/159 [01:39<00:00,  1.60it/s]


Epoch 1 | Train Loss: 1.2832 Acc: 0.5483 | Val Loss: 1.1643 Val Acc: 0.6090
  ✓ Best model saved (val_acc: 0.6090)


100%|██████████| 159/159 [01:20<00:00,  1.99it/s]


...

Epoch 25 | Train Loss: 0.6460 Acc: 0.7123 | Val Loss: 0.9221 Val Acc: 0.7357
  No improvement. Patience: 3/5


100%|██████████| 159/159 [01:21<00:00,  1.96it/s]


Epoch 26 | Train Loss: 0.6361 Acc: 0.7138 | Val Loss: 0.9289 Val Acc: 0.7375
  No improvement. Patience: 4/5


100%|██████████| 159/159 [01:20<00:00,  1.97it/s]


Epoch 27 | Train Loss: 0.6372 Acc: 0.7159 | Val Loss: 0.9099 Val Acc: 0.7288
  No improvement. Patience: 5/5
Early stopping triggered at epoch 27
Best Val Acc: 0.7375


In [37]:
from sklearn.metrics import recall_score
import torch
import numpy as np

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in tqdm(test_loader):
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Overall recall (macro = average recall across all classes)
recall_macro = recall_score(all_labels, all_preds, average='macro')
recall_micro = recall_score(all_labels, all_preds, average='micro')
recall_weighted = recall_score(all_labels, all_preds, average='weighted')

# Per-class recall
recall_per_class = recall_score(all_labels, all_preds, average=None)

print(f"Recall (Macro):    {recall_macro:.4f}")
print(f"Recall (Micro):    {recall_micro:.4f}")
print(f"Recall (Weighted): {recall_weighted:.4f}")
print(f"\nPer-Class Recall:")
for i, r in enumerate(recall_per_class):
    print(f"  Class {i}: {r:.4f}")




100%|██████████| 159/159 [01:30<00:00,  1.75it/s]


Recall (Macro):    0.6876
Recall (Micro):    0.7375
Recall (Weighted): 0.7375

Per-Class Recall:
  Class 0: 0.6667
  Class 1: 0.7757
  Class 2: 0.5865
  Class 3: 0.6842
  Class 4: 0.6141
  Class 5: 0.8169
  Class 6: 0.5368
  Class 7: 0.8200


In [40]:
import torch
file_path = "/content/best_model.pth"
model_state_dict = torch.load(file_path)
print(model_state_dict.values())

odict_values([tensor([[[[ 1.2156e-01,  6.5634e-01,  4.5671e-01],
          [-1.1092e-01, -6.1004e-01, -3.3345e-01],
          [ 2.7964e-02, -1.0312e-01, -1.0324e-01]],

         [[ 6.3553e-02,  1.6552e+00,  1.7436e+00],
          [-1.3646e-01, -1.5367e+00, -1.5937e+00],
          [ 5.0196e-02, -1.1360e-01, -1.2600e-01]],

         [[ 8.7276e-02,  3.6126e-01,  2.6946e-01],
          [-1.1966e-01, -2.8122e-01, -2.1883e-01],
          [ 3.6658e-02, -7.0751e-02, -8.1917e-02]]],


        [[[ 1.6449e-01, -2.0041e-01,  8.3092e-02],
          [ 8.9009e-01, -1.2110e+00,  2.7610e-01],
          [ 1.0740e+00, -1.2603e+00,  2.0645e-01]],

         [[ 3.2816e-01, -4.3449e-01,  1.8769e-01],
          [ 1.6213e+00, -2.1188e+00,  4.1014e-01],
          [ 1.7230e+00, -2.0756e+00,  3.3958e-01]],

         [[ 9.5290e-02, -1.5971e-01,  7.4559e-02],
          [ 8.0502e-01, -9.7034e-01,  2.6280e-01],
          [ 7.1944e-01, -1.0026e+00,  1.9870e-01]]],


        [[[ 5.3857e-02,  2.6367e-01,  1.1696e-01],
 